In [ ]:
from pyspark.sql import functions as F
from spark_session_config import spark
from schema import samples_data_schema

enriched_df = spark.readStream.format("kafka") \
.option("kafka.bootstrap.servers", "course-kafka:9092") \
.option("subscribe", "alert-data") \
.option("startingOffsets", "earliest") \
.load() \
.select(F.col("value").cast("string"))

parsed_data = enriched_df \
.withColumn("parsed_json",F.from_json(F.col("value"),samples_data_schema))\
.select(F.col('parsed_json.*'))

agg_df = parsed_data \
    .withWatermark("event_time", "15 minutes") \
    .groupBy(
        F.window(
            "event_time","15 minutes", "1 minute"
        )
    ) \
    .agg(
    F.count("*").alias("num_of_rows"),
    F.sum(F.when(F.lower(F.col("color_name"))=="black",1).otherwise(0)).alias("num_of_black"),
    F.sum(F.when(F.lower(F.col("color_name"))=="white",1).otherwise(0)).alias("num_of_white"),
    F.sum(F.when(F.lower(F.col("color_name"))=="silver",1).otherwise(0)).alias("num_of_silver"),
    F.max("speed").alias("max_speed"),
    F.max("gear").alias("max_gear"),
    F.max("rpm").alias("max_rpm")
    )

agg_df.writeStream \
.trigger(processingTime = '1 minute') \
.format("console") \
.outputMode('complete') \
.start() \
.awaitTermination()

spark.stop()

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/developer/.ivy2/cache
The jars for the packages stored in: /home/developer/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5ca4abd1-24ae-4d56-86e1-6cee09e896d2;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 630ms :: artifacts dl 12m

-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------+-----------+------------+------------+-------------+---------+--------+-------+
|              window|num_of_rows|num_of_black|num_of_white|num_of_silver|max_speed|max_gear|max_rpm|
+--------------------+-----------+------------+------------+-------------+---------+--------+-------+
|{2026-08-15 23:29...|        230|           0|           0|            0|      151|       2|   7872|
|{2026-08-15 23:25...|        230|           0|           0|            0|      151|       2|   7872|
|{2026-08-15 23:20...|        230|           0|           0|            0|      151|       2|   7872|
|{2026-08-15 23:17...|        230|           0|           0|            0|      151|       2|   7872|
|{2026-08-15 23:31...|        230|           0|           0|            0|      151|       2|   7872|
|{2026-08-15 23:22...|        230|           0|           0|            0|      151|   